# 04. Viewing a finished run

Step through validation tiles of a saved run: sort by error type, toggle labels,
predictions, errors and sigmoid output, and move the per-class thresholds to see how
IoU on each tile responds. Needs no GPU; it reads `val_outputs.npz` and `scores.json`
from the run folder. "save view" writes the current figure to the run's `figures/` folder.

## Setup

On Colab this mounts Google Drive, clones the repo and installs it. Locally it assumes
`pip install -e .` has been run in the repo and the tiles are in `data/chactun/`.

In [ ]:
REPO_URL = "https://github.com/skumagai3/maya-lidar.git"
DRIVE_DIR = "/content/drive/MyDrive/maya-lidar"           # Colab: runs are saved here
DATA_ZIP = f"{DRIVE_DIR}/data/22202395.zip"               # Colab: Figshare bundle, or a zip of the tile files

import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/maya-lidar")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))  # an editable install is not importable until restart
else:
    DATA_ZIP = None

import numpy as np
import matplotlib.pyplot as plt
from mayalidar import config as C, data as D, viz as V

paths = C.project_paths(DRIVE_DIR)
D.ensure_extracted(DATA_ZIP, paths.data_dir)
tiles = D.index_tiles(paths.data_dir)
print(f"{len(tiles)} tiles in {paths.data_dir}; runs go to {paths.runs_dir}")

## Choose a run (default: the latest run of the experiment)

In [ ]:
EXPERIMENT = "unet_scratch"   # or "resnet50_unet"
RUN = None                    # e.g. "20261113-091500"; None picks the latest full run

runs = sorted(d for d in (paths.runs_dir / EXPERIMENT).iterdir()
              if d.is_dir() and not d.name.endswith("smoke") and (d / "val_outputs.npz").exists())
run_dir = paths.runs_dir / EXPERIMENT / RUN if RUN else runs[-1]
print("viewing", run_dir)

In [ ]:
from mayalidar.viewer import Viewer
Viewer(run_dir, paths.data_dir).show()